In [ ]:
# %matplotlib inline
%matplotlib widget
%config InlineBackend.figure_format = 'retina'
from IPython.display import display, clear_output

In [ ]:
import random

import matplotlib.pyplot as plt
import pandas as pd
import pyspiel
from open_spiel.python.algorithms import outcome_sampling_mccfr
from tqdm.auto import trange

from cleptoninja import register_game as register_cleptoninja_game
from player import GreedyPlayer, Player, PolicyPlayer, RandomPlayer

In [ ]:
# TODO: Model players as a dataclass, hold their hands there. adapt rest of code
# TODO: Have auction resolution whithin dataclass. Query from game for card redistribution
# TODO: Change if phase ... for pattern matching everywhere

In [ ]:
register_cleptoninja_game()

## Train


In [ ]:
class FixedPlayersState(pyspiel.State):
    def __init__(self, game, wrapped_state, players: list[Player | None]):
        super().__init__(game)
        self._wrapped_state = wrapped_state
        self.player = players

    # Override to limit available actions to **the one** forced by the fixed player
    def _legal_actions(self, player_id=None) -> list[int]:
        player_id = player_id if player_id is not None else self.current_player()

        if self.is_terminal() or player_id != self.current_player():
            return []

        player = self.player[player_id]
        actions = (
            [player.action(self._wrapped_state)]
            if player
            else self._wrapped_state._legal_actions(player_id)
        )

        return actions

    # --- delegate to wrapped state ---
    def current_player(self) -> int:
        return self._wrapped_state.current_player()

    def _apply_action(self, action: int) -> None:
        return self._wrapped_state._apply_action(action)

    def is_terminal(self) -> bool:
        return self._wrapped_state.is_terminal()

    def returns(self) -> list[float]:
        return self._wrapped_state.returns()

    def information_state_string(self, player_id: int) -> str:
        return self._wrapped_state.information_state_string(player_id)

    def observation_string(self, player_id: int) -> str:
        return self._wrapped_state.observation_string(player_id)

    def __str__(self) -> str:
        return self._wrapped_state.__str__()

In [ ]:
class FixedPlayersGameWrapper(pyspiel.Game):
    def __init__(self, wrapped_game: pyspiel.Game, players: list[Player | None]):
        self.wrapped_game = wrapped_game
        self.players = players

        super().__init__(
            wrapped_game.get_type(),
            wrapped_game.game_info,
            wrapped_game.get_parameters(),
        )

    def new_initial_state(self):
        return FixedPlayersState(
            self, self.wrapped_game.new_initial_state(), self.players
        )

    def make_py_observer(self, iig_obs_type=None, params=None):
        return self.wrapped_game(iig_obs_type, params)

In [ ]:
def run_game(
    game,
    players=[RandomPlayer(id) for id in range(4)],
    debug=False,
):
    state = game.new_initial_state()
    while not state.is_terminal():
        player = state.current_player()
        action = players[player].action(state)
        state.apply_action(action)

        if debug:
            print(state)
            print()

    return state

In [ ]:
def ff(player_payouts: dict[str, list[int]]):
    game_count = max(len(x) for x in player_payouts.values())
    for player_name, payouts in player_payouts.items():
        missed_game_count = game_count - len(payouts)
        player_payouts[player_name] = [0] * missed_game_count + payouts

    pd.DataFrame(player_payouts).plot(subplots=True, sharex=True, sharey=True)

In [ ]:
def ff(player_payouts: dict[str, list[int]]):
    game_count = max((len(x) for x in player_payouts.values()), default=0)
    padded = {
        name: [0] * (game_count - len(payouts)) + list(payouts)
        for name, payouts in player_payouts.items()
    }
    df = pd.DataFrame(padded)
    df = df[sorted(df.columns)]
    clear_output(wait=True)
    display(df)

In [ ]:
# import matplotlib
# print("matplotlib backend:", matplotlib.get_backend())

In [ ]:
from collections import defaultdict
from statistics import mean, quantiles

PLAYER_COUNT = 4
OUTER_ITERATION_COUNT = 5 * PLAYER_COUNT  # 20
INNER_ITERATION_COUNT = 100_000  # 10_000
EVALUATION_ITERATION_COUNT = 5_000  # 5_000


# One pool per seat
fixed_players_pools = [[GreedyPlayer(i), RandomPlayer(i)] for i in range(PLAYER_COUNT)]

payout_stats = {
    "min": defaultdict(list),
    "q1": defaultdict(list),
    "median": defaultdict(list),
    "mean": defaultdict(list),
    "q3": defaultdict(list),
    "max": defaultdict(list),
}
base_game = pyspiel.load_game("clepto_ninja")
for outer_iteration_index in trange(OUTER_ITERATION_COUNT, desc="Outer loop"):
    # Train
    training_seat = outer_iteration_index % PLAYER_COUNT
    players = [
        None if seat == training_seat else random.choice(fixed_players_pools[seat])
        for seat in range(PLAYER_COUNT)
    ]

    game = FixedPlayersGameWrapper(base_game, players)
    solver = outcome_sampling_mccfr.OutcomeSamplingSolver(game)
    for _ in trange(INNER_ITERATION_COUNT, desc="Inner loop", leave=False):
        solver.iteration()

    trained_player = PolicyPlayer(training_seat, solver.average_policy())
    fixed_players_pools[training_seat].append(trained_player)

    evaluation_players_pools = [
        [GreedyPlayer(i), RandomPlayer(i), fixed_players_pools[i][-1]]
        for i in range(PLAYER_COUNT)
    ]

    # Eval
    player_payouts = defaultdict(list)
    for _ in trange(EVALUATION_ITERATION_COUNT, desc="Evaluation", leave=False):
        players = [random.choice(pool) for pool in evaluation_players_pools]
        end_state = run_game(game=base_game, players=players)

        for player, payout in zip(players, end_state.returns()):
            player_payouts[player.name].append(payout)

    for player_name, payouts in player_payouts.items():
        payouts = player_payouts[player_name]
        q1, q2, q3 = quantiles(payouts)
        payout_stats["min"][player_name].append(min(payouts))
        payout_stats["q1"][player_name].append(q1)
        payout_stats["median"][player_name].append(q2)
        payout_stats["mean"][player_name].append(mean(payouts))
        payout_stats["q3"][player_name].append(q3)
        payout_stats["max"][player_name].append(max(payouts))

    if outer_iteration_index % PLAYER_COUNT == PLAYER_COUNT - 1:
        # Print current scores after all seats have been trained
        ff(payout_stats["mean"])

In [ ]:
max_len = max(len(v) for players in payout_stats.values() for v in players.values())

def left_pad(lst, n):
    return [0.0] * (n - len(lst)) + lst

df = pd.concat(
    {
        stat: pd.DataFrame(
            {player: left_pad(values, max_len) for player, values in players.items()}
        )
        for stat, players in payout_stats.items()
    },
    axis=1,
    names=["stat", "player"],
).rename_axis(index="i")

# optional ordering
stat_order = ["min", "q1", "median", "mean", "q3", "max"]
df = df.reindex(stat_order, axis=1, level="stat")
df = df.sort_index(axis=1, level=["stat", "player"])

df

## Evaluate


In [ ]:
SIMULATION_COUNT = 100_000
ALL_MACHINE_PLAYERS = [
    GreedyPlayer,
    PolicyPlayer,
    RandomPlayer,
]
player_count = 4

rows = []
for _ in trange(SIMULATION_COUNT):
    players = [random.choice(pool) for pool in evaluation_players_pools]
    end_state = run_game(game=base_game, players=players)
    rows.append(end_state.returns() + [p.name for p in players])

game_results = pd.DataFrame(
    rows,
    columns=[f"payout_{i}" for i in range(player_count)]
    + [f"player_{i}" for i in range(player_count)],
)

In [ ]:
def plot_game_payouts(game_results, max_plotted_games=1_000):
    game_results = (
        game_results
        if len(game_results) < max_plotted_games
        else game_results.sample(max_plotted_games)
    )

    player_cols = [col for col in game_results if col.startswith("player_")]
    player_count = len(player_cols)
    _, ax = plt.subplots(figsize=(9, 3))

    # Build a stable color mapping for policies
    policies = pd.unique(game_results[player_cols].values.ravel())
    cmap = dict(zip(policies, plt.cm.tab10.colors[: len(policies)]))

    for i in range(player_count):
        ax.scatter(
            game_results.index,
            game_results[f"payout_{i}"],
            c=game_results[f"player_{i}"].map(cmap),
            s=20,
        )

    ax.set_xlabel("run")
    ax.set_ylabel("payout")
    ax.set_title(f"Player payouts ({len(game_results)} simulations)")

    # Optional legend
    handles = [
        plt.Line2D([0], [0], marker="o", linestyle="", color=c, label=p)
        for p, c in cmap.items()
    ]
    ax.legend(
        handles=handles, title="player", bbox_to_anchor=(1.02, 1), loc="upper left"
    )

    plt.tight_layout()
    plt.show()

In [ ]:
plot_game_payouts(game_results, max_plotted_games=1_000)

In [ ]:
def player_payout_stats(game_results):
    player_cols = [col for col in game_results if col.startswith("player_")]
    payout_cols = [col for col in game_results if col.startswith("payout_")]
    player_count = len(player_cols)

    long_df = pd.concat(
        [
            game_results[[payout_cols[i], player_cols[i]]].rename(
                columns={payout_cols[i]: "payout", player_cols[i]: "player"}
            )
            for i in range(player_count)
        ],
        ignore_index=True,
    )

    summary = (
        long_df.groupby("player")["payout"]
        .agg(
            min="min",
            q1=lambda x: x.quantile(0.25),
            median="median",
            mean="mean",
            q3=lambda x: x.quantile(0.75),
            max="max",
        )
        .reset_index()
    )

    return summary

In [ ]:
player_payout_stats(game_results)

In [ ]:
game_results[[c for c in game_results.columns if "payout_" in c]].describe()